# Alfabetização na região de Santa Cruz do Sul: Censo 2010 x Censo 2022

Compara a alfabetização nos **16 municípios da microrregião de Santa Cruz do Sul** (IBGE), com o **Rio Grande do Sul** e o **Brasil** como referência:
taxa de alfabetização de quem tem 15 anos ou mais, número de pessoas que não sabem ler e escrever, faixas de idade, sexo e cor ou raça.

Tudo vem da API do SIDRA (apisidra.ibge.gov.br). O download leva cerca de 1 minuto.

**No fim, o notebook baixa um arquivo `alfabetizacao_santa_cruz_2010_2022.zip`. Mande esse zip de volta** (ele tem a planilha com todos os números exatos e os gráficos).

In [ ]:
import re, time, os, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.float_format', '{:,.1f}'.format)
pd.set_option('display.max_rows', 300); pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})

MICRO = '43020'   # microrregião de Santa Cruz do Sul (IBGE)
FOCO = 'Santa Cruz do Sul'
REFERENCIAS = {'RS': 'n3/43', 'Brasil': 'n1/1'}
NIVEIS = {f'n6/in n9 {MICRO}': None, **{nivel: nome for nome, nivel in REFERENCIAS.items()}}
COL_LOCAL = ['Município (Código)', 'Unidade da Federação (Código)', 'Brasil (Código)']
API = 'https://apisidra.ibge.gov.br/values'
_cache = {}
MUNICIPIOS = {}   # preenchido pela primeira consulta: código -> nome

def sidra(caminho, tentativas=4):
    url = f'{API}/{caminho.strip("/")}'.replace(' ', '%20')
    if url in _cache:
        return _cache[url]
    for i in range(tentativas):
        try:
            r = requests.get(url, timeout=180)
            if r.status_code == 400:
                raise ValueError(f'consulta inválida: {r.text[:200]}')
            r.raise_for_status()
            dados = r.json()
            df = pd.DataFrame(dados[1:]).rename(columns=dados[0])
            df['Valor'] = pd.to_numeric(df['Valor'], errors='coerce')
            _cache[url] = df
            return df
        except ValueError:
            raise
        except Exception as e:
            print(f'  tentativa {i+1} falhou: {e}')
            time.sleep(2 ** (i + 1))
    raise RuntimeError(url)

def por_local(consulta):
    """Mesma consulta para os municípios da microrregião, o RS e o Brasil, com a coluna 'local'."""
    partes = []
    for nivel, nome in NIVEIS.items():
        df = sidra(f'{consulta}/{nivel}').copy()
        col = next(c for c in COL_LOCAL if c in df.columns)
        if nome is None:
            nomes = df['Município'].str.replace(r'\s*(-\s*RS|\(RS\))$', '', regex=True)
            MUNICIPIOS.update(dict(zip(df[col], nomes)))
            df['local'] = df[col].map(MUNICIPIOS)
        else:
            df['local'] = nome
        partes.append(df)
    return pd.concat(partes, ignore_index=True)

def br(x, casas=1):
    if pd.isna(x) or np.isinf(x):
        return 's/d'
    return f'{x:,.{casas}f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

def idade_num(rotulo):
    """'15 anos' -> 15; 'Menos de 1 ano' -> 0; '65 anos ou mais' -> '65+'; outros grupos -> None"""
    if rotulo == 'Menos de 1 ano':
        return 0
    m = re.fullmatch(r'(\d+) anos?( ou mais)?', rotulo)
    if not m:
        return None
    if m.group(2):
        return {'100': 100, '65': '65+'}.get(m.group(1))
    return int(m.group(1))

def por_idade(df):
    """Tabela local x idade simples (e o grupo '65+' quando existir)."""
    col = next(c for c in df.columns if df[c].astype(str).eq('15 anos').any())
    d = df.assign(idade=df[col].map(idade_num)).dropna(subset=['idade'])
    return d.groupby(['local', 'idade'], sort=False)['Valor'].sum()

GRUPOS = {'15 a 24 anos': range(15, 25), '25 a 44 anos': range(25, 45), '45 a 64 anos': range(45, 65)}
def agrupa(s):
    """Soma idades simples nos grupos; 65+ vem do grupo pronto (2022) ou da soma das idades (2010)."""
    out = {}
    for local in s.index.get_level_values(0).unique():
        x = s.loc[local]
        r = {g: x[[i for i in ids if i in x.index]].sum() for g, ids in GRUPOS.items()}
        r['65 anos ou mais'] = x['65+'] if '65+' in x.index else x[[i for i in x.index if isinstance(i, (int, float, np.integer, np.floating)) and i >= 65]].sum()
        r['15 anos ou mais'] = sum(r.values())
        out[local] = r
    return pd.DataFrame(out).T

os.makedirs('graficos', exist_ok=True)
teste = por_local('t/9542/v/950/p/2022/c2/6794/c86/95251/c287/100362/c59/93024')
LOCAIS = sorted(MUNICIPIOS.values()) + list(REFERENCIAS)
MUNIS = sorted(MUNICIPIOS.values())
print(len(MUNIS), 'municípios:', ', '.join(MUNIS))

## 1. Taxa de alfabetização e pessoas que não sabem ler e escrever (15 anos ou mais)

In [ ]:
# 2022: alfabetizadas e total por idade (tabela 9542)
B22 = 't/9542/v/950/p/2022/c2/{sexo}/c86/{cor}/c287/all/c59/{alf}'
def censo22(sexo='6794', cor='95251'):
    alf = agrupa(por_idade(por_local(B22.format(sexo=sexo, cor=cor, alf='1023'))))
    tot = agrupa(por_idade(por_local(B22.format(sexo=sexo, cor=cor, alf='93024'))))
    return alf, tot

# 2010: alfabetizadas por idade (tabela 3213) e população por idade (tabela 1378)
alf10 = agrupa(por_idade(por_local('t/3213/v/2591/p/2010/c287/all')))
tot10 = agrupa(por_idade(por_local('t/1378/v/93/p/2010/c1/0/c2/0/c455/0/c287/all')))
alf22, tot22 = censo22()

taxa10 = (100 * alf10 / tot10).reindex(LOCAIS)
taxa22 = (100 * alf22 / tot22).reindex(LOCAIS)
analf10 = (tot10 - alf10).reindex(LOCAIS)
analf22 = (tot22 - alf22).reindex(LOCAIS)

G = '15 anos ou mais'
res = pd.DataFrame({
    'Taxa 2010 (%)': taxa10[G], 'Taxa 2022 (%)': taxa22[G], 'Variação (pontos)': taxa22[G] - taxa10[G],
    'Não sabem ler 2010': analf10[G], 'Não sabem ler 2022': analf22[G],
    'Variação (%)': 100 * (analf22[G] / analf10[G] - 1),
    'Taxa de analfabetismo 2022 (%)': 100 - taxa22[G],
})
res['Posição 2022 (1 = maior taxa)'] = res.loc[MUNIS, 'Taxa 2022 (%)'].rank(ascending=False, method='min')
res['Posição no avanço (1 = maior)'] = res.loc[MUNIS, 'Variação (pontos)'].rank(ascending=False, method='min')
res = res.sort_values('Taxa 2022 (%)', ascending=False)
print(res.round(1).to_string())

## 2. Por faixa de idade

In [ ]:
print('TAXA DE ALFABETIZAÇÃO POR IDADE, 2010')
print(taxa10.round(1).to_string())
print('\nTAXA DE ALFABETIZAÇÃO POR IDADE, 2022')
print(taxa22.round(1).to_string())
print('\nPESSOAS QUE NÃO SABEM LER E ESCREVER, 2022, POR IDADE')
print(analf22.round(0).to_string())
print('\n% DOS QUE NÃO SABEM LER QUE TÊM 65 ANOS OU MAIS (2022)')
print((100 * analf22['65 anos ou mais'] / analf22[G]).round(1).to_string())

## 3. Por sexo e por cor ou raça (só 2022)

In [ ]:
alf_h, tot_h = censo22(sexo='4')
alf_m, tot_m = censo22(sexo='5')
alf_b, tot_b = censo22(cor='2776')
alf_n, tot_n = censo22(cor='2777,2779')
grupos = pd.DataFrame({
    'Homens (%)': 100 * alf_h[G] / tot_h[G], 'Mulheres (%)': 100 * alf_m[G] / tot_m[G],
    'Brancos (%)': 100 * alf_b[G] / tot_b[G], 'Negros, pretos e pardos (%)': 100 * alf_n[G] / tot_n[G],
    'Negros de 15+ (base)': tot_n[G],
    'Homens 65+ (%)': 100 * alf_h['65 anos ou mais'] / tot_h['65 anos ou mais'],
    'Mulheres 65+ (%)': 100 * alf_m['65 anos ou mais'] / tot_m['65 anos ou mais'],
}).reindex(LOCAIS)
grupos['Diferença brancos - negros (pontos)'] = grupos['Brancos (%)'] - grupos['Negros, pretos e pardos (%)']
print(grupos.round(1).to_string())

## 4. Gráficos

In [ ]:
COR_FOCO, COR_VIZ, COR_2010, COR_REF = '#c0392b', '#1f5f99', '#b8b8b8', '#555555'

def halteres(a, b, titulo, arquivo, sufixo='%', casas=1, locais=MUNIS + ['RS']):
    d = pd.DataFrame({'2010': a.reindex(locais), '2022': b.reindex(locais)}).dropna().sort_values('2022')
    fig, ax = plt.subplots(figsize=(8, 0.36 * len(d) + 1.4))
    for i, (local, r) in enumerate(d.iterrows()):
        cor = COR_FOCO if local == FOCO else COR_REF if local in REFERENCIAS else COR_VIZ
        ax.plot([r['2010'], r['2022']], [i, i], color=cor, lw=2, alpha=.35)
        ax.scatter(r['2010'], i, s=45, color='white', edgecolor=COR_2010, lw=2, zorder=2)
        ax.scatter(r['2022'], i, s=60, color=cor, edgecolor='white', lw=1.2, zorder=3)
        ax.annotate(f" {br(r['2022'], casas)}{sufixo}", (max(r['2010'], r['2022']), i), va='center', fontsize=8.5,
                    xytext=(5, 0), textcoords='offset points')
    ax.set_yticks(range(len(d))); ax.set_yticklabels(d.index)
    for t in ax.get_yticklabels():
        if t.get_text() == FOCO: t.set_color(COR_FOCO); t.set_weight('bold')
    ax.grid(axis='x', color='#e6e6e6'); ax.set_axisbelow(True); ax.spines['left'].set_visible(False)
    ax.tick_params(axis='y', length=0); ax.margins(x=.1)
    ax.set_title(titulo, loc='left', weight='bold')
    plt.tight_layout(); plt.savefig(f'graficos/{arquivo}.png', dpi=200, bbox_inches='tight'); plt.show()

halteres(taxa10[G], taxa22[G], 'Taxa de alfabetização, 15 anos ou mais (%), 2010 e 2022', '01_taxa')

d = res.loc[MUNIS, 'Taxa de analfabetismo 2022 (%)'].sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(d.index, d.values, color=[COR_FOCO if l == FOCO else COR_VIZ for l in d.index])
for i, v in enumerate(d.values): ax.annotate(f' {br(v)}%', (v, i), va='center', fontsize=8.5)
ax.axvline(res.loc['RS', 'Taxa de analfabetismo 2022 (%)'], color=COR_REF, ls='--', lw=1)
ax.set_title('Pessoas de 15 anos ou mais que não sabem ler e escrever, 2022 (%)\n(linha tracejada: RS)', loc='left', weight='bold')
plt.tight_layout(); plt.savefig('graficos/02_analfabetismo_2022.png', dpi=200, bbox_inches='tight'); plt.show()

fig, ax = plt.subplots(figsize=(9, 7))
h = taxa22.loc[MUNIS + ['RS'], list(GRUPOS) + ['65 anos ou mais']]
im = ax.imshow(h.values, cmap='Blues', vmin=70, vmax=100, aspect='auto')
ax.set_xticks(range(h.shape[1])); ax.set_xticklabels(h.columns)
ax.set_yticks(range(h.shape[0])); ax.set_yticklabels(h.index)
for i in range(h.shape[0]):
    for j in range(h.shape[1]):
        v = h.values[i, j]; ax.text(j, i, br(v), ha='center', va='center', fontsize=8, color='white' if v > 92 else 'black')
ax.set_title('Taxa de alfabetização por idade, 2022 (%)', loc='left', weight='bold')
plt.tight_layout(); plt.savefig('graficos/03_idade_2022.png', dpi=200, bbox_inches='tight'); plt.show()

halteres(grupos['Negros, pretos e pardos (%)'], grupos['Brancos (%)'],
         'Alfabetização 2022: negros (círculo) e brancos (ponto), 15+ (%)', '04_cor_2022')

In [ ]:
# Mapa: taxa de alfabetização 2022
import geopandas as gpd
malha = gpd.read_file('https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-43-mun.json')
malha['id'] = malha['id'].astype(str)
reg = malha[malha.id.isin(MUNICIPIOS)].copy()
reg['taxa'] = reg['id'].map(MUNICIPIOS).map(taxa22[G])
x0, y0, x1, y1 = reg.total_bounds
fig, ax = plt.subplots(figsize=(8, 8))
malha.plot(ax=ax, color='#f0f0f0', edgecolor='white', lw=.6)
reg.plot(ax=ax, column='taxa', cmap='Blues', vmin=88, vmax=100, edgecolor='white', lw=1, legend=True,
         legend_kwds={'label': 'Taxa de alfabetização 2022 (%)', 'shrink': .6})
for _, m in reg.iterrows():
    p = m.geometry.representative_point()
    ax.annotate(f"{MUNICIPIOS[m.id]}\n{br(m.taxa)}%", (p.x, p.y), ha='center', fontsize=7.5)
ax.set_xlim(x0 - .05, x1 + .05); ax.set_ylim(y0 - .05, y1 + .05); ax.set_axis_off()
ax.set_title('Alfabetização na microrregião de Santa Cruz do Sul, 2022', loc='left', weight='bold')
plt.tight_layout(); plt.savefig('graficos/00_mapa.png', dpi=200, bbox_inches='tight'); plt.show()

## 5. Salvar e baixar (mande o zip de volta)

In [ ]:
with pd.ExcelWriter('alfabetizacao_santa_cruz_2010_2022.xlsx') as xls:
    res.round(2).to_excel(xls, sheet_name='Resumo 15+')
    taxa10.round(2).to_excel(xls, sheet_name='Taxa por idade 2010')
    taxa22.round(2).to_excel(xls, sheet_name='Taxa por idade 2022')
    analf10.to_excel(xls, sheet_name='Não sabem ler 2010')
    analf22.to_excel(xls, sheet_name='Não sabem ler 2022')
    tot10.to_excel(xls, sheet_name='População 2010')
    tot22.to_excel(xls, sheet_name='População 2022')
    grupos.round(2).to_excel(xls, sheet_name='Sexo e cor 2022')
with zipfile.ZipFile('alfabetizacao_santa_cruz_2010_2022.zip', 'w') as z:
    for arq in ['alfabetizacao_santa_cruz_2010_2022.xlsx'] + [f'graficos/{g}' for g in sorted(os.listdir('graficos'))]:
        z.write(arq)
print('Salvo: alfabetizacao_santa_cruz_2010_2022.zip')
try:
    from google.colab import files
    files.download('alfabetizacao_santa_cruz_2010_2022.zip')
except ImportError:
    pass